# Steps 1 to 3 on MEVID-SC (same clothes only)

Reruns the CSCI baseline on the **same-clothes dataset** (`Data/MEVID-SameClothes`).

**Why no 30-minute GPU run:** CSCI is frozen, so each clip's feature depends only on that clip's own frames. Every MEVID-SC test clip is an identical copy of an original MEVID test clip (`original_index_test.txt` links them), so cell 2 takes the CSCI features already saved on Drive and keeps only the same-clothes clips. It checks person, outfit and camera clip by clip. Optional cell 5 reruns CSCI directly on the new folder as an independent confirmation.

| Cell | What it does | Time |
|---|---|---|
| 1 | Mount Drive, find the dataset and saved features | 1 min |
| 2 | Same-clothes features + accuracy (mAP, Rank-1) | 1 min |
| 3 | Step 1: frozen protocol (same person folds as before) | seconds |
| 4 | Step 3: CSCI-alone baselines, compared with clothing-change MEVID | 2 min |
| 5 | Optional: rerun CSCI on the new folder (A100, ~45 min) | optional |

Cells 1 to 4 need no GPU.

In [ ]:
# 1. Mount Drive; find MEVID-SameClothes and the saved CSCI features
import os, glob, json, numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')

def find(patterns):
    for pat in patterns:
        hits = sorted(glob.glob(pat, recursive=True))
        if hits:
            return os.path.dirname(hits[0])
    raise FileNotFoundError(patterns[0])

SD = '/content/drive/Shareddrives/*'
SC_DIR = find([f'{SD}/Data/MEVID-SameClothes/SUBSET_REPORT.json',
               f'{SD}/**/MEVID-SameClothes/SUBSET_REPORT.json'])
CSCI_OUT = find([f'{SD}/Data/MEVID-results/CSCI/test_templates_csci.npz',
                 f'{SD}/**/CSCI/test_templates_csci.npz'])
SC_OUT = os.path.join(os.path.dirname(CSCI_OUT), 'CSCI-SameClothes')
os.makedirs(SC_OUT, exist_ok=True)
print('Dataset (same clothes):', SC_DIR)
print('CSCI features (MEVID) :', CSCI_OUT)
print('New results go to     :', SC_OUT)


In [ ]:
# 2. CSCI features for the MEVID-SC test set + accuracy
z = np.load(f'{CSCI_OUT}/test_templates_csci.npz')
T0, pid0, oid0, cam0 = z['templates'], z['pid'], z['oid'], z['cam']
A = f'{SC_DIR}/annotations'
info = np.loadtxt(f'{A}/track_test_info.txt').astype(np.int64)
orig = np.loadtxt(f'{A}/original_index_test.txt').astype(np.int64)
qidx = np.loadtxt(f'{A}/query_IDX.txt').astype(np.int64)
gidx = np.setdiff1d(np.arange(len(info)), qidx)

# each MEVID-SC clip must be the same clip (person, outfit, camera)
# as its original MEVID row
assert len(orig) == len(info) and orig.max() < len(T0)
assert (pid0[orig] == info[:, 2]).all(), 'person mismatch'
assert (oid0[orig] == info[:, 3]).all(), 'outfit mismatch'
assert (cam0[orig] == info[:, 4]).all(), 'camera mismatch'
T, pid, oid, cam = T0[orig], info[:, 2], info[:, 3], info[:, 4]
assert (pd.Series(oid).groupby(pid).nunique() == 1).all(), \
    'a person has more than one outfit'
print(f'Clip check PASSED: {len(info)} clips, {len(np.unique(pid))} people,'
      f' one outfit each; {len(qidx)} queries, {len(gidx)} gallery clips')


def ap_cmc(order, good, junk):
    order = order[~np.isin(order, junk)]
    hits = np.isin(order, good)
    cmc = np.zeros(len(order)); cmc[np.argmax(hits):] = 1.0
    r = np.flatnonzero(hits)
    return np.mean((np.arange(len(r)) + 1) / (r + 1)), cmc


def evaluate(T, pid, cam, qidx, gidx):
    """Standard video re-ID scoring: same person on the same camera
    is ignored; a correct match must come from another camera."""
    S = T[qidx] @ T[gidx].T
    aps, cmcs = [], []
    for i, q in enumerate(qidx):
        sp = np.flatnonzero(pid[gidx] == pid[q])
        sc = np.flatnonzero(cam[gidx] == cam[q])
        good, junk = np.setdiff1d(sp, sc), np.intersect1d(sp, sc)
        if good.size:
            a, c = ap_cmc(np.argsort(-S[i]), good, junk)
            aps.append(a); cmcs.append(c[:20])
    cmc = np.mean(cmcs, 0)
    return {'queries': len(aps), 'mAP %': round(100 * np.mean(aps), 1),
            'Rank-1 %': round(100 * cmc[0], 1),
            'Rank-5 %': round(100 * cmc[4], 1),
            'Rank-10 %': round(100 * cmc[9], 1)}


acc = pd.DataFrame({
    'MEVID (with clothing change)':
        evaluate(T0, pid0, cam0, z['qidx'], z['gidx']),
    'MEVID-SC (same clothes)': evaluate(T, pid, cam, qidx, gidx)}).T
print('\nCSCI accuracy (frozen released MEVID checkpoint)')
display(acc)
acc.to_csv(f'{SC_OUT}/accuracy_csci_sc.csv')
np.savez(f'{SC_OUT}/test_templates_csci_sc.npz', templates=T, pid=pid,
         oid=oid, cam=cam, qidx=qidx, gidx=gidx, orig_idx=orig)
print('Saved test_templates_csci_sc.npz and accuracy_csci_sc.csv to',
      SC_OUT)


In [ ]:
# 3. STEP 1 on MEVID-SC: frozen protocol.
#    Reuses the SAME person folds frozen for MEVID (seed 2027), so nothing
#    is re-chosen after seeing results. Never overwritten once saved.
from datetime import date
old = json.load(open(f'{CSCI_OUT}/PROTOCOL_frozen.json'))
fold_of = {int(p): f for p, f in old['folds'].items()}
fold = np.array([fold_of[int(p)] for p in pid])


def describe(f):
    ps = np.unique(pid[fold == f])
    ncam = [len(np.unique(cam[pid == p])) for p in ps]
    return {'people': int(len(ps)), 'clips': int((fold == f).sum()),
            'queries': int((fold[qidx] == f).sum()),
            'mean cameras per person': round(float(np.mean(ncam)), 2)}


summary = {f: describe(f) for f in 'AB'}
display(pd.DataFrame(summary).T)

path = f'{SC_OUT}/PROTOCOL_frozen_sc.json'
if os.path.exists(path):
    saved = json.load(open(path))
    assert saved['folds'] == {str(int(p)): fold_of[int(p)]
                              for p in np.unique(pid)}, 'folds changed!'
    print('Protocol already frozen on', saved['frozen_on'], '- unchanged.')
else:
    protocol = {
        'frozen_on': str(date.today()),
        'dataset': ('MEVID-SC: MEVID v1 test split restricted to one outfit '
                    'per person (54 ids, 1,368 clips, 231 queries, 11 '
                    'cameras); Data/MEVID-SameClothes'),
        'backbone': ('CSCI (ICCV 2025) released MEVID video checkpoint '
                     'ez_eva02_vid_hybrid_extra, color profile 17; frozen. '
                     'Note: trained by its authors on the full MEVID train '
                     'split, which includes clothing changes.'),
        'features': ('CSCI clip features from the full MEVID run, '
                     'restricted via original_index_test.txt (identical '
                     'clips; features are per clip)'),
        'accuracy_metrics': 'mAP, Rank-1/5/10 on the 231 MEVID-SC queries',
        'calibration_scheme': old['calibration_scheme'],
        'seed': old['seed'],
        'folds_source': ('identical person folds frozen on '
                         f"{old['frozen_on']} for MEVID; reused unchanged"),
        'folds': {str(int(p)): fold_of[int(p)] for p in np.unique(pid)},
        'fold_summary': summary,
        'calibrated_quantities': old['calibrated_quantities'],
        'frozen_before_test_use': old['frozen_before_test_use'],
        'primary_outcomes': [
            'consistent-ID IDF1 and people-per-ID at matched precision',
            'risk-coverage: share of identity links resolved at 95% '
            'precision',
            'calibration fidelity: predicted vs observed precision on the '
            'evaluation fold',
            'cross-camera recall'],
    }
    with open(path, 'w') as f:
        json.dump(protocol, f, indent=2)
    print('Protocol frozen and saved to', path)


In [ ]:
# 4. STEP 3 on MEVID-SC: CSCI-alone baselines, person-disjoint 2-fold
#    cross-fitting (calibrate on one fold's people, freeze, evaluate on
#    the other fold; swap). No GPU, about 2 minutes.
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.optimize import linear_sum_assignment

z = np.load(f'{SC_OUT}/test_templates_csci_sc.npz')
T, pid, cam = z['templates'], z['pid'], z['cam']
qidx, gidx = z['qidx'], z['gidx']
proto = json.load(open(f'{SC_OUT}/PROTOCOL_frozen_sc.json'))
fold = np.array([proto['folds'][str(int(p))] for p in pid])
TARGETS = [0.95, 0.90]                      # frozen precision targets


# ---------------- (A) match decisions: accept or abstain ----------------
def match_rows(f):
    qs = qidx[fold[qidx] == f]; gs = gidx[fold[gidx] == f]
    Sf = T[qs] @ T[gs].T
    rows = []
    for i, q in enumerate(qs):
        cand = np.flatnonzero(cam[gs] != cam[q])
        if cand.size < 2 or not np.any(pid[gs][cand] == pid[q]):
            continue
        s = Sf[i, cand]; o = np.argsort(-s)
        b = gs[cand[o[0]]]
        rows.append({'best': float(s[o[0]]),
                     'margin': float(s[o[0]] - s[o[1]]),
                     'correct': bool(pid[b] == pid[q])})
    return pd.DataFrame(rows)


def pick_threshold(df, score, target):
    best = None
    for t in np.unique(df[score])[::-1]:
        acc = df[df[score] >= t]
        if len(acc) and acc.correct.mean() >= target:
            best = t
    return best


def apply(df, score, t):
    acc = df[df[score] >= t] if t is not None else df.iloc[:0]
    wrong = (~df.correct).sum()
    return {'coverage %': 100 * len(acc) / len(df),
            'precision %': 100 * acc.correct.mean() if len(acc) else np.nan,
            'silent errors': int((~acc.correct).sum()),
            'errors caught %': 100 * (wrong - (~acc.correct).sum())
                               / max(1, wrong)}


rows = []
for cal, ev in [('A', 'B'), ('B', 'A')]:
    dc, de = match_rows(cal), match_rows(ev)
    for score in ['best', 'margin']:
        for tg in TARGETS:
            t = pick_threshold(dc, score, tg)
            c, e = apply(dc, score, t), apply(de, score, t)
            rows.append({
                'calibrate->evaluate': f'{cal}->{ev}',
                'confidence': {'best': 'top-1 similarity',
                               'margin': 'top-1 minus top-2'}[score],
                'target %': int(100 * tg),
                'predicted precision %': round(c['precision %'], 1),
                'observed precision %': round(e['precision %'], 1),
                'coverage %': round(e['coverage %'], 1),
                'silent errors': e['silent errors'],
                'errors caught %': round(e['errors caught %'], 1),
                'top-1 correct % (no rule)':
                    round(100 * de.correct.mean(), 1)})
match = pd.DataFrame(rows)
match.to_csv(f'{SC_OUT}/step3_match_decisions_csci_sc.csv', index=False)
print('(A) MATCH DECISIONS on MEVID-SC, CSCI alone (held-out fold)')
display(match)


# ---------------- (B) consistent ID across cameras ----------------
def score_ids(pred, p, c):
    P = {x: i for i, x in enumerate(np.unique(p))}
    C = {x: j for j, x in enumerate(np.unique(pred))}
    M = np.zeros((len(P), len(C)))
    for a, b in zip(p, pred):
        M[P[a], C[b]] += 1
    r, cc = linear_sum_assignment(-M)
    sp = p[:, None] == p[None, :]; sc = pred[:, None] == pred[None, :]
    off = ~np.eye(len(p), dtype=bool); tp = (sp & sc & off).sum()
    xc = sp & (c[:, None] != c[None, :])
    return {'IDF1 %': 100 * M[r, cc].sum() / len(p),
            'IDs per person': (M > 0).sum(1).mean(),
            'people per ID': (M > 0).sum(0).mean(),
            'pair precision %': 100 * tp / max(1, (sc & off).sum()),
            'pair recall %': 100 * tp / max(1, (sp & off).sum()),
            'cross-camera recall %': 100 * (xc & sc).sum() / max(1, xc.sum())}


def tree(idx):
    d = 1 - np.clip(T[idx] @ T[idx].T, -1, 1)
    return linkage(d[np.triu_indices(len(idx), 1)], 'average')


SIMS = np.round(np.arange(0.10, 0.991, 0.01), 2)
rows = []
for cal, ev in [('A', 'B'), ('B', 'A')]:
    ic, ie = np.flatnonzero(fold == cal), np.flatnonzero(fold == ev)
    Zc, Ze = tree(ic), tree(ie)
    sweep = pd.DataFrame([dict(s=s, **score_ids(
        fcluster(Zc, 1 - s, 'distance'), pid[ic], cam[ic])) for s in SIMS])
    points = {'max IDF1': float(sweep.loc[sweep['IDF1 %'].idxmax(), 's'])}
    for tg in TARGETS:
        ok = sweep[sweep['pair precision %'] >= 100 * tg]
        points[f'pair precision >= {int(100 * tg)}%'] = (
            float(ok['s'].min()) if len(ok) else float(SIMS.max()))
    for name, s in points.items():
        pc = sweep.loc[sweep.s == s, 'pair precision %'].iloc[0]
        res = score_ids(fcluster(Ze, 1 - s, 'distance'), pid[ie], cam[ie])
        rows.append({'calibrate->evaluate': f'{cal}->{ev}',
                     'operating point': name, 'link sim >=': s,
                     'predicted pair precision %': round(pc, 1),
                     **{k: round(v, 2) for k, v in res.items()}})
cid = pd.DataFrame(rows)
cid.to_csv(f'{SC_OUT}/step3_consistent_id_csci_sc.csv', index=False)
print('\n(B) CONSISTENT ID on MEVID-SC, CSCI alone (held-out fold)')
display(cid)


# ------------- side by side: clothing change vs same clothes -------------
def both(path_old, df_new, keys, cols):
    out = {'MEVID-SC (same clothes)':
           df_new.groupby(keys).mean(numeric_only=True)[cols]}
    if os.path.exists(path_old):
        out['MEVID (with clothing change)'] = (
            pd.read_csv(path_old).groupby(keys)
            .mean(numeric_only=True)[cols])
    return pd.concat(out, axis=1).round(1)


print('\nCOMPARISON (mean of both fold directions)')
display(both(f'{CSCI_OUT}/step3_match_decisions_csci.csv', match,
             ['confidence', 'target %'],
             ['observed precision %', 'coverage %', 'silent errors']))
display(both(f'{CSCI_OUT}/step3_consistent_id_csci.csv', cid,
             ['operating point'],
             ['IDF1 %', 'IDs per person', 'people per ID',
              'pair precision %', 'cross-camera recall %']))
print('Saved step3_*_sc.csv to', SC_OUT)


## Optional: rerun CSCI directly on the MEVID-SameClothes folder

Not needed for the results above. Run it once if you want written proof that the new folder works on its own, and that the features from cell 2 match a fresh run. Needs an **A100** runtime, about 10 minutes to unpack the 14.5 GB test archive and about 30 minutes of CSCI. Set `RUN_FULL_CSCI = True` first.

In [ ]:
# 5. OPTIONAL: independent CSCI run on MEVID-SC (A100, ~45 minutes)
RUN_FULL_CSCI = False

import subprocess, sys
if RUN_FULL_CSCI:
    LOCAL, ROOT = '/content/mevid_sc', '/content/mevid_sc_root'
    os.makedirs(LOCAL, exist_ok=True); os.makedirs(ROOT, exist_ok=True)
    if not os.path.isdir(f'{LOCAL}/bbox_test'):
        print('Unpacking the MEVID-SC test archive (about 10 minutes) ...')
        subprocess.run(['tar', '-xf', f'{SC_DIR}/mevid-sc-bbox-test.tar',
                        '-C', LOCAL], check=True)
    for n in ['train_name.txt', 'test_name.txt', 'track_train_info.txt',
              'track_test_info.txt', 'query_IDX.txt']:
        if not os.path.exists(f'{ROOT}/{n}'):
            os.symlink(f'{SC_DIR}/annotations/{n}', f'{ROOT}/{n}')
    if not os.path.exists(f'{ROOT}/bbox_test'):
        os.symlink(f'{LOCAL}/bbox_test', f'{ROOT}/bbox_test')

    REPO = '/content/ICCV-CSCI-Person-ReID'
    if not os.path.isdir(REPO):
        subprocess.run(['git', 'clone', '-q', 'https://github.com/ppriyank/'
                        'ICCV-CSCI-Person-ReID.git', REPO], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'timm==1.0.9', 'einops', 'yacs', 'mmengine', 'decord',
                    'tensorboardX', 'tensorboard', 'termcolor', 'tabulate',
                    'h5py', 'scikit-learn', 'lmdb', 'albumentations',
                    'torchinfo', 'grad-cam', 'opencv-python-headless'],
                   check=True)
    WT = f'{CSCI_OUT}/ez_eva02_vid_hybrid_extra_best.pth'
    assert os.path.exists(WT), 'CSCI weights not found in ' + CSCI_OUT

    # same two patches as the MEVID run: feature dump + eval-only loading
    f = f'{REPO}/processor/eval_mevid.py'; src = open(f).read()
    anchor = '    distmat = distmat.cpu()\n'
    hook = (anchor +
            "    if local_rank == 0 and os.environ.get('CSCI_DUMP'):\n"
            "        import numpy as _np\n"
            "        _np.savez(os.environ['CSCI_DUMP'], "
            "qf=qf.float().cpu().numpy(), gf=gf.float().cpu().numpy(), "
            "q_pids=q_pids.numpy(), g_pids=g_pids.numpy(), "
            "q_camids=q_camids.numpy(), g_camids=g_camids.numpy())\n")
    if 'CSCI_DUMP' not in src:
        src = src.replace(anchor, hook)
        if '\nimport os' not in src:
            src = 'import os\n' + src
        open(f, 'w').write(src)
    f = f'{REPO}/data/dataset_loader.py'; src = open(f).read()
    old_l = ('        self.return_index = return_index\n'
             '        self.__getitem__(0)\n')
    new_l = ('        self.return_index = return_index\n'
             '        if not train:\n            self.__getitem__(0)\n')
    if old_l in src:
        open(f, 'w').write(src.replace(old_l, new_l))

    DUMP = f'{SC_OUT}/csci_sc_rerun_features.npz'
    env = dict(os.environ, CSCI_DUMP=DUMP, PYTHONPATH=REPO)
    cmd = ['torchrun', '--nproc_per_node=1', '--master_port=12356',
           'train_two_step.py', '--eval', '--resume',
           '--config_file', 'configs/mevid_eva02_l_cloth.yml',
           'DATA.ROOT', ROOT + '/', 'TEST.WEIGHT', WT,
           'SOLVER.SEED', '12345', 'TRAIN.TRAIN_VIDEO', 'True',
           'DATA.DATASET', 'mevid', 'MODEL.NAME',
           'ez_eva02_vid_hybrid_extra', 'TRAIN.COLOR_PROFILE', '17',
           'DATA.TEST_BATCH', '32']
    if not os.path.exists(DUMP):
        with open(f'{SC_OUT}/csci_sc_rerun_log.txt', 'w') as log:
            p = subprocess.Popen(cmd, cwd=REPO, env=env, text=True,
                                 stdout=subprocess.PIPE,
                                 stderr=subprocess.STDOUT)
            for line in p.stdout:
                log.write(line)
                if any(k in line for k in ['top1', 'mAP', 'Error',
                                           'Traceback', 'loaded']):
                    print(line.rstrip())
            p.wait()
    assert os.path.exists(DUMP), 'CSCI stopped before dumping; paste log'

    r = np.load(DUMP)
    z = np.load(f'{SC_OUT}/test_templates_csci_sc.npz')
    n = lambda x: x / np.linalg.norm(x, axis=1, keepdims=True)
    cq = (n(r['qf']) * z['templates'][z['qidx']]).sum(1)
    cg = (n(r['gf']) * z['templates'][z['gidx']]).sum(1)
    print(f'Fresh run vs cell 2 features, cosine: queries min '
          f'{cq.min():.4f}, gallery min {cg.min():.4f} (1.0 = identical)')
else:
    print('Skipped (set RUN_FULL_CSCI = True to run).')
